In [5]:
import json
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
import pandas as pd

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Load biến môi trường từ file .env
load_dotenv(r"D:\AI\.env")
print("GOOGLE_API_KEY present:", bool(os.getenv("GOOGLE_API_KEY")))

c:\Users\Admin\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\miniconda3\envs\rag\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Python-dotenv could not parse statement starting at line 5


GOOGLE_API_KEY present: True


In [6]:
# 1. Load dữ liệu từ file JSON
with open("D:\\AI\\assistant\\data\\dulieu.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [7]:
documents = []
for item in data["articles"]:
    for chunk in item["chunks"]:
        # Mỗi chunk là một Document, kèm metadata (nguồn, tiêu đề)
        doc = Document(
            page_content=chunk,
            metadata={"source": item["url"], "title": item["title"]}
        )
        documents.append(doc)

print(f"Tổng số chunks cần index: {len(documents)}")

Tổng số chunks cần index: 198


In [8]:
# 3. Tạo Vector Database
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(documents, embeddings)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19088\2022950839.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
c:\Users\Admin\miniconda3\envs\rag\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Bạn là một chatbot thời sự chuyên nghiệp.

Nhiệm vụ của bạn là trả lời câu hỏi CHỈ dựa trên thông tin trong context.
KHÔNG suy đoán, KHÔNG bịa.

Nếu không có thông tin phù hợp, hãy trả lời:
"Hiện tại tôi chưa tìm thấy thông tin phù hợp trong dữ liệu."

Context:
{context}

Câu hỏi:
{question}

Trả lời (tiếng Việt, ngắn gọn, trung lập):
""")


In [11]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_output_tokens=1024
    
)

In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [14]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
def answer_question(question: str) -> str:
    return rag_chain.invoke(question)

In [24]:
print(
    answer_question(
        "tin tức không khí lạnh"
    )
)

Khoảng gần sáng và ngày 31/1, không khí lạnh sẽ ảnh hưởng đến khu vực Đông Bắc Bộ, sau đó ảnh hưởng đến Bắc Trung Bộ, khu vực Tây Bắc Bộ và Trung Trung Bộ.

Đêm 30 và ngày 31/1, khu vực Bắc Bộ và Bắc Trung Bộ có mưa nhỏ rải rác. Từ ngày 31/1, Đông Bắc Bộ trời chuyển rét, vùng núi có nơi rét đậm; từ đêm 31/1, Bắc Trung Bộ và Tây Bắc Bộ (trừ Điện Biên và Lai Châu) trời chuyển rét.

Nhiệt độ thấp nhất trong đợt không khí lạnh này ở Bắc Bộ và Bắc Trung Bộ phổ biến 12-15 độ C, vùng núi Bắc Bộ 9-12 độ C, vùng núi cao có nơi dưới 8 độ C.

Tại Hà Nội, đêm 30 và ngày 31/1 có mưa, mưa nhỏ rải rác, từ ngày 31/1 trời chuyển rét. Nhiệt độ thấp nhất phổ biến 13-15 độ C.

Do ảnh hưởng của không khí lạnh, từ ngày 1/2 đến đêm 3/2, khu vực từ Quảng Trị đến Đà Nẵng và phía Đông các tỉnh từ Quảng Ngãi đến Gia Lai có mưa, mưa rào rải rác và có nơi có dông.
